# Esercitazione software 2

## Filtraggio di segnali musicali campionati 

Autori:

- Fissolo Emanuele - *s323585*
- Flora Alessandro - *s321504*
- Giraudo Giacomo - *s321784*
- Intagliata Francesco - *s325961*

### 0) Operazioni preliminari
In questa cella vengono eseguite le operazioni preliminari necessarie al funzionamento del codice

In [ ]:
# --- ESEGUIRE SOLO SU COLAB ---
%cd /content
!pip install numpy matplotlib soundfile scipy
!git clone https://github.com/AlexF1789/homeworkTes.git
%cd homeworkTes
!git checkout homework2
# ------------------------------

In [ ]:
#in questo blocco eseguiamo tutte le operazione preliminari necessarie per il funzionamento dei
#blocchi di codice successivi

import numpy as np
import matplotlib.pyplot as plt
import tes, soundfile, os, math
from tes import Tipo_filtro
from IPython.display import Audio, display

# creiamo la cartella output se questa non esiste
os.makedirs('output', exist_ok=True)

file_CornfieldChase = str(os.path.join('input', 'CornfieldChase.oga'))
file_BohemianRhapsody = str(os.path.join('input', 'QueenBohemianRhapsody.oga'))


### 1) Filtraggio dei segnali

#### Cornfield Chase

##### **Lettura del file audio**

Procediamo a leggere il file audio e a ricavare alcuni parametri utili in seguito

In [ ]:
audio, fs = soundfile.read(file_CornfieldChase)
audio = np.array(audio[:, 0], dtype=np.float32)

len_sec = len(audio)/fs

Df = 1/(len(audio)*1/fs)
x_axis_f = [x*Df for x in range(int(-len(audio)/2),int(len(audio)/2))]
x_axis_t = [x for x in range(0, len(audio))]

print(f"Estratti {len(audio)} campioni - campionati a {fs/1000:.1f}kHz - durata {len_sec} secondi - risoluzione in frequenza {Df:.2f}Hz")

Iniziamo il nostro studio sul filtraggio dei segnali a partire dalla traccia audio **Cornfield Chase** (presa dalla colonna sonora del film **Interstellar**). Questo brano è caratterizzato da singole note di pianoforte ripetute, con una forte base di bassi

##### **Spettro del segnale originale**

Procediamo a plottare lo spettro in frequenza della traccia audio originale scelta

In [ ]:
fft_py = np.abs(np.fft.fft(audio))

# calcoliamo il limite di frequenza per avere la banda al 99% del segnale
max_freq_banda = tes.get_limite_banda(tes.get_spettro(fft_py), Df, 99)
print(f"La frequenza corrispondente alla banda al 99% è {max_freq_banda}Hz")

# plottiamo la trasformata
plt.plot(x_axis_f, np.fft.fftshift(fft_py))
plt.grid(True)
plt.axis([-max_freq_banda*1.25, max_freq_banda*1.25, 0, 1.25 * np.max(fft_py)])
plt.xlabel("Frequency [Hz]")
plt.ylabel("Amplitude")
plt.title("Cornfield Chase\nSpettro del segnale originale")
#plt.savefig("CornfieldChase_originale.png")
plt.show()

##### **Anteprima del file audio**
La seguente cella consente di ottenere un'anteprima del file audio riproducendolo - la sua mancata riproduzione non inficia sul resto dell'elaborazione

In [ ]:
display(Audio(file_CornfieldChase))

##### **Prova applicazione filtro "Porta"**

Generiamo un filtro "porta discreta nel tempo", che si comporta come un filtro passa-basso in frequenza. Vogliamo creare questo filtro in modo che abbia una frequenza di taglio a 3dB attorno alla frequenza 500Hz.
Abbiamo deciso di passare ad una frequenza di taglio a 500Hz, in quanto se tagliassimo a 1KHz non avremmo un effetto molto tangibile in quanto la maggior parte delle note usate in questo brano si trovano sotto a questa frequenza

In [ ]:
T = tes.get_T_per_taglio(500, Tipo_filtro.PORTA, 0)
h_t = tes.get_porta_discreta(T, 0, fs, int(T*fs))
h_t = h_t / sum(h_t)

audio_filtrato = np.convolve(audio, h_t, mode = "same")
fft_py_filtrato = np.abs(np.fft.fft(audio_filtrato))
y_lim = 1.25 * np.max(fft_py_filtrato)

plt.plot(x_axis_f, np.fft.fftshift(fft_py_filtrato))
plt.grid(True)
plt.axis([-max_freq_banda*1.25, max_freq_banda*1.25, 0, y_lim])
plt.xlabel("Frequency [Hz]")
plt.ylabel("Amplitude")
plt.title("Cornfield Chase\nSpettro del segnale dopo l'applicazione " \
"del filtro \"Porta\"")
#plt.savefig("CornfieldChase_porta.png")
plt.show()

Come possiamo vedere dal grafico dello spettro in frequenza della traccia audio filtrata e come possiamo anche sentire ascoltando la traccia filtrata stessa, il filtro definito come una porta nel tempo ha uno scarso effetto in frequenza. Questo accade poichè la porta nel tempo si comporta come una sinc in frequenza, che decresce in maniera abbastanza lenta e successivamente oscilla andando a zero sono in alcune frequenze. 

Proviamo ad ascoltare il nuovo segnale audio

In [ ]:
file_out = str(os.path.join('output', 'CornfieldChaseFilter1.wav'))
soundfile.write(file=file_out, data=audio_filtrato, samplerate=fs)

display(Audio(file_out))

Non particolarmente soddisfatti del risultato dal punto di visto grafico proviamo dunque a effettuare un plot su **scala logaritmica** dello spettro del segnale originale e dello spettro del segnale filtrato.

In [ ]:
plt.semilogy(x_axis_f, np.fft.fftshift(fft_py), 'r', label='Spettro originale')
plt.semilogy(x_axis_f, np.fft.fftshift(fft_py_filtrato), 'g', label='Spettro filtrato')
plt.grid(True)
plt.xlim([-max_freq_banda*1.25, max_freq_banda*1.25])
plt.legend(loc='lower left')
plt.xlabel("Frequency [Hz]")
plt.ylabel("Amplitude")
plt.title("Cornfield Chase\nSpettro logaritmico dopo l'applicazione " \
"del filtro \"Porta\"")
#plt.savefig("CornfieldChase_porta_log.png")
plt.show()

##### **Prova applicazione filtro "Coseno rialzato"**

Generiamo un filtro "coseno rialzato nel tempo" con beta = 0.5, che si comporta sempre come un filtro passa-basso.

In [ ]:
T = tes.get_T_per_taglio(500, Tipo_filtro.COS_RIALZATO, 0.5)
h_t = tes.get_coseno_rialzato(20*T, T, 0, fs, int(20*T*fs), 0.5)
h_t = h_t / sum(h_t)

audio_filtrato = np.convolve(audio, h_t, mode = "same")
fft_py_filtrato = np.abs(np.fft.fft(audio_filtrato))

plt.plot(x_axis_f, np.fft.fftshift(fft_py_filtrato))
plt.grid(True)
plt.axis([-max_freq_banda*1.25, max_freq_banda*1.25, 0, y_lim])
plt.xlabel("Frequency [Hz]")
plt.ylabel("Amplitude")
plt.title("Cornfield Chase\nSpettro del segnale dopo l'applicazione " \
"del filtro \"Coseno rialzato\"")
#plt.savefig("CornfieldChase_coseno_rialzato.png")
plt.show()

Proviamo ad ascoltare il nuovo segnale audio

In [ ]:
file_out = str(os.path.join('output', 'CornfieldChaseFilter2.wav'))
soundfile.write(file=file_out, data=audio_filtrato, samplerate=fs)

display(Audio(file_out))

In questo secondo caso possiamo invece notare un effetto molto più marcato del filtro sulla traccia audio. Impostando una frequenza di taglio a 500Hz (per escludere una buona parte delle frequenze più alte della nostra traccia audio) vediamo un notevole abbassamento nell'ampiezza di queste frequenze e ascoltando la traccia audio filtrata possiamo sentire un suono molto più ovattato, con i bassi estremamente più preponderanti rispetto alle frequenze più alte, che in certi casi risultano quasi completamente assenti

##### **Prova applicazione filtro "Coseno rialzato passa-alto"**

Generiamo un filtro di tipo passa-alto a partire dal filtro a coseno rialzato del punto precedente.
Per gli stessi motivi del punto precedente sceglieremo anche qui una frequenza di taglio a 500Hz al posto di quella suggerita di 1KHz

In [ ]:
T = tes.get_T_per_taglio(500, Tipo_filtro.COS_RIALZATO, 0.5)
h_t = tes.get_cr_passa_alto(20*T, T, fs, 0.5)

audio_filtrato = np.convolve(audio, h_t, mode = "same")
fft_py_filtrato = np.abs(np.fft.fft(audio_filtrato))

plt.plot(x_axis_f, np.fft.fftshift(fft_py_filtrato))
plt.grid(True)
plt.axis([-max_freq_banda*1.25, max_freq_banda*1.25, 0, y_lim])
plt.xlabel("Frequency [Hz]")
plt.ylabel("Amplitude")
plt.title("Cornfield Chase\nSpettro del segnale dopo l'applicazione " \
"del filtro \"Passa-alto\"")
#plt.savefig("CornfieldChase_passa_alto.png")
plt.show()

Proviamo ad ascoltare la traccia audio filtrata ottenuta

In [ ]:
file_out = str(os.path.join('output', 'CornfieldChaseFilter3.wav'))
soundfile.write(file=file_out, data=audio_filtrato, samplerate=fs)

display(Audio(file_out))

In questo terzo caso infine notiamo che otteniamo l'effetto opposto rispetto al filtro precedente. Infatti qui possiamo osservare (sia dallo spettro plottato, sia ascoltando la traccia audio filtrata) una pesante attenuazione delle frequenze basse delle spettro sonoro. Impostando la frequenza di taglio a 500Hz otteniamo un suono privo dei bassi, che all'orecchio umano risulta poco "corposo"

#### Bohemian Rhapsody

##### **Lettura del file audio**

Procediamo a leggere il file audio e a ricavare alcuni parametri utili in seguito

In [ ]:
audio, fs = soundfile.read(file_BohemianRhapsody)
audio = np.array(audio[:, 0], dtype=np.float32)

len_sec = len(audio)/fs

Df = 1/(len(audio)*1/fs)
x_axis_f = [x*Df for x in range(int(-len(audio)/2),int(len(audio)/2)+1)]
x_axis_t = [x for x in range(0, len(audio))]

print(f"Estratti {len(audio)} campioni - campionati a {fs/1000:.1f}kHz - durata {len_sec} secondi - risoluzione in frequenza {Df:.2f}Hz")

Adesso proviamo a effettuare lo stesso studio di prima sulla traccia audio **Bohemian Rhapsody** (20 secondi presi dal brano **Bohemian Rhapsody** dei Queen). Questo brano è caratterizzato da un assolo di chitarra elettrica composto da tantissime note diverse, ha una forte base di bassi e inoltre è presente la voce del cantante che aggiunge numerose frequenze alte

##### **Spettro del segnale originale**

Procediamo a plottare lo spettro in frequenza della traccia audio originale scelta

In [ ]:
fft_py = np.abs(np.fft.fft(audio))

# calcoliamo il limite di frequenza per avere la banda al 99% del segnale
max_freq_banda = tes.get_limite_banda(tes.get_spettro(fft_py), Df, 99)
print(f"La frequenza corrispondente alla banda al 99% è {max_freq_banda}Hz")

# plottiamo la trasformata
plt.plot(x_axis_f, np.fft.fftshift(fft_py))
plt.grid(True)
plt.axis([-max_freq_banda*1.25, max_freq_banda*1.25, 0, 1.25 * np.max(fft_py)])
plt.xlabel("Frequency [Hz]")
plt.ylabel("Amplitude")
plt.title("Bohemian Rhapsody\nSpettro del segnale originale")
#plt.savefig("BohemianRhapsody_originale.png")
plt.show()

##### **Anteprima del file audio**
La seguente cella consente di ottenere un'anteprima del file audio riproducendolo - la sua mancata riproduzione non inficia sul resto dell'elaborazione

In [ ]:
display(Audio(file_BohemianRhapsody))

##### **Prova applicazione filtro "Porta"**

Generiamo un filtro "porta discreta nel tempo", che si comporta come un filtro passa-basso in frequenza. Vogliamo creare questo filtro in modo che abbia una frequenza di taglio a 3dB attorno alla frequenza 500Hz.
Anche per questo secondo caso abbiamo deciso di passare ad una frequenza di taglio a 500Hz, in quanto se tagliassimo a 1KHz non avremmo un effetto molto tangibile in quanto la maggior parte delle note usate in questo brano si trovano sotto a questa frequenza

In [ ]:
T = tes.get_T_per_taglio(500, Tipo_filtro.PORTA, 0)
h_t = tes.get_porta_discreta(T, 0, fs, int(T*fs))
h_t = h_t / sum(h_t)

audio_filtrato = np.convolve(audio, h_t, mode = "same")
fft_py_filtrato = np.abs(np.fft.fft(audio_filtrato))
y_lim = 1.25 * np.max(fft_py_filtrato)

plt.plot(x_axis_f, np.fft.fftshift(fft_py_filtrato))
plt.grid(True)
plt.axis([-max_freq_banda*1.25, max_freq_banda*1.25, 0, y_lim])
plt.xlabel("Frequency [Hz]")
plt.ylabel("Amplitude")
plt.title("Bohemian Rhapsody\nSpettro del segnale dopo l'applicazione " \
"del filtro \"Porta\"")
#plt.savefig("BohemianRhapsody_porta.png")
plt.show()

Anche in questo caso possiamo notare che questo tipo di filtro ha uno scarso effetto sullo spettro del segnale. Tuttavia grazie allo spettro più "pieno" di questo brano riusciamo a vedere come l'effetto di attenuazione del filtro va a seguire i lobi della sinc. Lo spettro, al di fuori della banda principale della sinc, ha una serie di lobi interrotti dalle poche frequenze che vengono portate totalmente a zero dalla sinc

In [ ]:
file_out = str(os.path.join('output', 'BohemianRhapsodyFilter1.wav'))
soundfile.write(file=file_out, data=audio_filtrato, samplerate=fs)

display(Audio(file_out))

##### **Prova applicazione filtro "Coseno rialzato"**

Generiamo un filtro "coseno rialzato nel tempo" con beta = 0.5, che si comporta sempre come un filtro passa-basso

In [ ]:
T = tes.get_T_per_taglio(500, Tipo_filtro.COS_RIALZATO, 0.5)
h_t = tes.get_coseno_rialzato(20*T, T, 0, fs, int(20*T*fs), 0.5)
h_t = h_t / sum(h_t)

audio_filtrato = np.convolve(audio, h_t, mode = "same")
fft_py_filtrato = np.abs(np.fft.fft(audio_filtrato))

plt.plot(x_axis_f, np.fft.fftshift(fft_py_filtrato))
plt.grid(True)
plt.axis([-max_freq_banda*1.25, max_freq_banda*1.25, 0, y_lim])
plt.xlabel("Frequency [Hz]")
plt.ylabel("Amplitude")
plt.title("Bohemian Rhapsody\nSpettro del segnale dopo l'applicazione " \
"del filtro \"Coseno rialzato\"")
#plt.savefig("BohemianRhapsody_coseno_rialzato.png")
plt.show()

Anche in questo caso il filtro a “coseno rialzato” ha avuto un effetto estremamente marcato, sono rimaste solo le frequenze centrali dello spettro, mentre quelle alte sono state completamente eliminate.

In [ ]:
file_out = str(os.path.join('output', 'BohemianRhapsodyFilter2.wav'))
soundfile.write(file=file_out, data=audio_filtrato, samplerate=fs)

display(Audio(file_out))

Ascoltando la traccia audio si può sentire che tutte le tonalità alte sono state eliminate (la voce del cantante è totalmente assente e quasi tutte le note della chitarra sono state tagliate), si sente solamente la batteria e una sorta di “rumore” dato dalle note basse della chitarra.

##### **Prova applicazione filtro "Coseno rialzato passa-alto"**

Generiamo un filtro di tipo passa-alto a partire dal filtro a coseno rialzato del punto precedente

In [ ]:
T = tes.get_T_per_taglio(500, Tipo_filtro.COS_RIALZATO, 0.5)
h_t = tes.get_cr_passa_alto(20*T, T, fs, 0.5)

audio_filtrato = np.convolve(audio, h_t, mode = "same")
fft_py_filtrato = np.abs(np.fft.fft(audio_filtrato))

plt.plot(x_axis_f, np.fft.fftshift(fft_py_filtrato))
plt.grid(True)
plt.axis([-max_freq_banda*1.25, max_freq_banda*1.25, 0, y_lim])
plt.xlabel("Frequency [Hz]")
plt.ylabel("Amplitude")
plt.title("Bohemian Rhapsody\nSpettro del segnale dopo l'applicazione " \
"del filtro \"Passa-alto\"")
#plt.savefig("BohemianRhapsody_passa_alto.png")
plt.show()

L’effetto del filtro passa alto in frequenza è anche qui notevole. Notiamo infatti come sia presente un buco tra le frequenze 0 e 500Hz, mentre le altre frequenze sono state lasciate intatte.

In [ ]:
file_out = str(os.path.join('output', 'BohemianRhapsodyFilter3.wav'))
soundfile.write(file=file_out, data=audio_filtrato, samplerate=fs)

display(Audio(file_out))

Ascoltando la traccia audio sentiamo che il suono è “metallico”, come se uscisse dalla cornetta di un vecchio telefono. Sentiamo chiaramente la mancanza della base bassa e questa mancanza rende il suono vuoto.

### 2) Calcolo della funzione di trasferimento

Proviamo ora a calcolare la funzione di trasferimento H(f) di un segnale al quale è stato applicato un filtro noto.

#### Funzione di trasferimento del filtro "Porta nel tempo"
In particolare proveremo ad applicare un filtro **porta nel tempo** a un segnale generato come **rumore bianco**. L'idea di fondo è che, tralasciando eventuale rumore, la trasformata in frequenza del filtro dovrà equivalere alla funzione di trasferimento calcolata come rapporto tra il segnale in uscita dal filtro e il segnale in ingresso (questi ultimi espressi ovviamente in frequenza).

In [ ]:
# creiamo il segnale rumore con la stessa frequenza di campionamento
# dei segnali audio usati fino ad ora
T = tes.get_T_per_taglio(1_000, tes.Tipo_filtro.PORTA, 0)
noise = tes.get_rumore_bianco(len(audio)/fs, fs)

# creiamo il filtro nel tempo
porta = tes.get_porta_discreta(T, 0, fs, math.ceil(T*fs))
porta = porta / np.sum(porta)

# calcoliamo l'uscita e la trasformata del filtro
uscita = np.convolve(noise, porta)
n_fft = uscita.size
trasf_filtro = np.abs(np.fft.fftshift(np.fft.fft(porta, n=n_fft)))

# calcoliamo le trasformate dell'ingresso e dell'uscita per calcolare
noise_f = np.fft.fftshift(np.fft.fft(noise, n=n_fft))
uscita_f = np.fft.fftshift(np.fft.fft(uscita, n=n_fft))

funz_trasf = tes.calcola_funzione_trasferimento(noise_f, uscita_f)

# plottiamo ora per verificare il risultato
Df = 1/(len(funz_trasf)*1/fs)
x_axis = [x*Df for x in range(math.ceil(-len(funz_trasf)/2),math.ceil(len(funz_trasf)/2))]
plt.plot(x_axis, funz_trasf, 'r', label='Funzione di \ntrasferimento')
plt.plot(x_axis, trasf_filtro, 'c--', label='Trasformata del \nfiltro')

plt.xlim(x_axis[len(x_axis)//4], x_axis[len(x_axis)//4*3])
plt.title('Funzione di trasferimento filtro \"Porta\"')
plt.legend(loc='upper left')

#plt.savefig("Funzione_trasferimento_porta.png")
plt.show()

#### Funzione di trasferimento del filtro "Coseno rialzato"
Proviamo ora ad applicare la stessa tecnica anche al filtro **coseno rialzato**, sempre usando come segnale in ingresso il **rumore bianco**

In [ ]:
# creiamo il segnale rumore con la stessa frequenza di campionamento
# dei segnali audio usati fino ad ora
beta = 0.5
T = tes.get_T_per_taglio(1000, tes.Tipo_filtro.COS_RIALZATO, beta)
noise = tes.get_rumore_bianco(len(audio)/fs, fs)

# creiamo il filtro nel tempo
cos = tes.get_coseno_rialzato(20*T, T, 0, fs, math.ceil(20*T*fs), beta)
cos = cos / np.sum(cos)

# calcoliamo l'uscita e la trasformata del filtro
uscita = np.convolve(noise, cos)
n_fft = uscita.size
trasf_filtro = np.abs(np.fft.fftshift(np.fft.fft(cos, n=n_fft)))

# calcoliamo le trasformate dell'ingresso e dell'uscita per calcolare
noise_f = np.fft.fftshift(np.fft.fft(noise, n=n_fft))
uscita_f = np.fft.fftshift(np.fft.fft(uscita, n=n_fft))

funz_trasf = tes.calcola_funzione_trasferimento(noise_f, uscita_f)

# plottiamo ora per verificare il risultato
Df = 1/(len(funz_trasf)*1/fs)
x_axis = [x*Df for x in range(math.ceil(-len(funz_trasf)/2),math.ceil(len(funz_trasf)/2))]
plt.plot(x_axis, funz_trasf, 'r', label='Funzione di \ntrasferimento')
plt.plot(x_axis, trasf_filtro, 'c--', label='Trasformata del \nfiltro')

plt.xlim(x_axis[len(x_axis)//4], x_axis[len(x_axis)//4*3])
plt.title('Funzione di trasferimento filtro \"Coseno rialzato\"')
plt.legend(loc='upper left')

#plt.savefig("Funzione_trasferimento_coseno_rialzato.png")
plt.show()

#### Funzione di trasferimento del filtro "Coseno rialzato passa-alto"
Proviamo infine ad applicare la stessa tecnica anche al filtro **coseno rialzato passa-alto**, sempre usando come segnale in ingresso il **rumore bianco**

In [ ]:
# creiamo il segnale rumore con la stessa frequenza di campionamento
# dei segnali audio usati fino ad ora
beta = 0.5
T = tes.get_T_per_taglio(1000, tes.Tipo_filtro.COS_RIALZATO, beta)
noise = tes.get_rumore_bianco(len(audio)/fs, fs)

# creiamo il filtro nel tempo
cos = tes.get_cr_passa_alto(20*T, T, fs, beta)

# calcoliamo l'uscita e la trasformata del filtro
uscita = np.convolve(noise, cos)
n_fft = uscita.size
trasf_filtro = np.abs(np.fft.fftshift(np.fft.fft(cos, n=n_fft)))

# calcoliamo le trasformate dell'ingresso e dell'uscita per calcolare
noise_f = np.fft.fftshift(np.fft.fft(noise, n=n_fft))
uscita_f = np.fft.fftshift(np.fft.fft(uscita, n=n_fft))

funz_trasf = tes.calcola_funzione_trasferimento(noise_f, uscita_f)

# plottiamo ora per verificare il risultato
Df = 1/(len(funz_trasf)*1/fs)
x_axis = [x*Df for x in range(math.ceil(-len(funz_trasf)/2),math.ceil(len(funz_trasf)/2))]
plt.plot(x_axis, funz_trasf, 'r', label='Funzione di \ntrasferimento')
plt.plot(x_axis, trasf_filtro, 'c--', label='Trasformata del \nfiltro')

plt.xlim(x_axis[len(x_axis)//4], x_axis[len(x_axis)//4*3])
plt.title('Funzione di trasferimento filtro \n\"Coseno rialzato passa-alto\"')
plt.legend(loc='lower left')

#plt.savefig("Funzione_trasferimento_passa_alto.png")
plt.show()